# MIMII Anomalib DataModule

This notebook uses a custom Lightning DataModule that reads `data/dcase-2020-spectrogram/meta.json` generated by  `mimii-toy.py` script.

In [2]:
import importlib
import sys
from pathlib import Path

from anomalib.engine import Engine

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.mimii_anomalib_datamodule as mimii_dm_module
importlib.reload(mimii_dm_module)
MIMIIAnomalibDataModule = mimii_dm_module.MIMIIAnomalibDataModule

In [3]:
from torchvision.transforms.v2 import Compose, Resize, Transform, ToTensor
dm = MIMIIAnomalibDataModule(
    root="../data/dcase-2020-spectrogram",
    categories=("fan", "pump", "slider", "valve"),
    train_phases=("id_00",),
    val_phases=("id_04",),
    test_phases=("id_02",),
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=8,
    augmentations=Compose([
        Resize((256, 256)),
        ToTensor()]
    ),
)

dm.setup("fit")
train_batch = next(iter(dm.train_dataloader()))
print("train keys:", train_batch.keys(include_none=False))
print("train image shape:", train_batch.image.shape)
print("train label unique:", train_batch.gt_label.unique())

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


train keys: ['image', 'gt_label', 'image_path']
train image shape: torch.Size([32, 3, 256, 256])
train label unique: tensor([False,  True])


In [3]:
from anomalib.models import Padim
model = Padim(backbone="resnet18")
engine = Engine()
engine.fit(model=model, datamodule=dm)
engine.test(model=model, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configur

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor  │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator     │      0 │ train │     0 │
│ 3 │ model          │ PadimModel    │  2.8 M │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric AUROC was called before the ``update`` method which may lead to 
errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric F1Score was called before the ``update`` method which may lead to 
errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.39416739344596863    │
│       image_F1Score       │    0.33502936363220215    │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 0.39416739344596863, 'image_F1Score': 0.33502936363220215}]

In [7]:
import os
# Disable tqdm progress bars to avoid recursion in Jupyter
os.environ['TQDM_DISABLE'] = '1'

# Disable rich rendering to avoid circular buffer issues
os.environ['ANOMALIB_DISABLE_RICH'] = '1'

from anomalib.models import Patchcore
model = Patchcore()
engine = Engine(logger=None)  # Disable loggers to prevent rendering issues
engine.fit(model=model, datamodule=dm)
engine.test(model=model, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/f74134118/anaconda3/envs/anomalib-env/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:534: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indices.:   0%|          | 0/481381 [00:00<?, ?it/s]
Selecting Coreset Indice

RecursionError: maximum recursion depth exceeded in comparison